In [ ]:
from pathlib import Path
import os
import shutil
import json
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

def find_project_root(start: Path | None = None) -> Path:
    """Walk upward until the repository root is found."""
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_MIDI_DIR = DATA_DIR / "raw_midi"
PROCESSED_DIR = DATA_DIR / "processed"
TRAIN_TEST_SPLIT_DIR = DATA_DIR / "train_test_split"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
GENERATED_MIDI_DIR = OUTPUT_DIR / "generated_midis"
PLOTS_DIR = OUTPUT_DIR / "plots"
SERVEY_RESULTS_DIR = OUTPUT_DIR / "servey_results"
LEGACY_SURVEY_DIR = OUTPUT_DIR / "survey_results"

for d in [
    DATA_DIR,
    RAW_MIDI_DIR,
    PROCESSED_DIR,
    TRAIN_TEST_SPLIT_DIR,
    OUTPUT_DIR,
    GENERATED_MIDI_DIR,
    PLOTS_DIR,
    SERVEY_RESULTS_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)


if not LEGACY_SURVEY_DIR.exists():
    try:
        LEGACY_SURVEY_DIR.symlink_to(SERVEY_RESULTS_DIR, target_is_directory=True)
    except Exception:
        LEGACY_SURVEY_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("RAW_MIDI_DIR:", RAW_MIDI_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("TRAIN_TEST_SPLIT_DIR:", TRAIN_TEST_SPLIT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("GENERATED_MIDI_DIR:", GENERATED_MIDI_DIR)
print("PLOTS_DIR:", PLOTS_DIR)
print("SERVEY_RESULTS_DIR:", SERVEY_RESULTS_DIR)


: 

In [ ]:
CSV_PATH = Path("/kaggle/input/datasets/talhaisl4m/midi-genre-datase/maestro-v3.0.0.csv")
if not CSV_PATH.exists():
    raise FileNotFoundError(f"Could not find the MAESTRO metadata CSV at: {CSV_PATH}")

metadata = pd.read_csv(CSV_PATH)
metadata.columns = [c.strip() for c in metadata.columns]

def pick_column(candidates):
    for name in candidates:
        if name in metadata.columns:
            return name
    return None

MIDI_COL = pick_column(["midi_filename", "midi_path", "midi_file", "filename", "path"])
SPLIT_COL = pick_column(["split", "subset", "partition"])
YEAR_COL = pick_column(["year", "performance_year"])
DURATION_COL = pick_column(["duration", "length_seconds", "seconds"])
COMPOSER_COL = pick_column(["canonical_composer", "composer"])
TITLE_COL = pick_column(["canonical_title", "title"])

if MIDI_COL is None:
    raise ValueError(f"Could not identify the MIDI filename column. Available columns: {list(metadata.columns)}")

print("Loaded metadata:", metadata.shape)
print("Detected columns:")
print("  MIDI_COL    =", MIDI_COL)
print("  SPLIT_COL   =", SPLIT_COL)
print("  YEAR_COL    =", YEAR_COL)
print("  DURATION_COL=", DURATION_COL)
print("  COMPOSER_COL=", COMPOSER_COL)
print("  TITLE_COL   =", TITLE_COL)


SEARCH_ROOTS = [
    CSV_PATH.parent,
    *CSV_PATH.parent.parents,
    Path("/kaggle/input"),
    PROJECT_ROOT,
]

def locate_midi_file(midi_ref: str) -> Path | None:
    """Return the first matching MIDI file for a relative path or basename."""
    if pd.isna(midi_ref):
        return None

    rel = Path(str(midi_ref).strip().replace("\\", "/"))
    candidates = []

    if rel.is_absolute():
        candidates.append(rel)


    for root in SEARCH_ROOTS:
        candidates.append(root / rel)
        candidates.append(root / rel.name)

    for cand in candidates:
        if cand.exists() and cand.is_file():
            return cand.resolve()

    for root in SEARCH_ROOTS:
        if not root.exists() or not root.is_dir():
            continue
        for cand in root.rglob(rel.name):
            if cand.is_file() and str(cand).replace("\\", "/").endswith(str(rel).replace("\\", "/")):
                return cand.resolve()

    return None

def stage_file(source: Path, target: Path) -> str:
    """Create a symlink when possible, otherwise copy the file into the target location."""
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        return "exists"
    try:
        target.symlink_to(source)
        return "symlink"
    except Exception:
        shutil.copy2(source, target)
        return "copy"

manifest_rows = []
missing_rows = []

MAX_FILES = int(os.environ.get("MAX_FILES", "0"))  # 0 = use the full metadata file
rows_iter = metadata.itertuples(index=False)

for i, row in enumerate(tqdm(rows_iter, total=len(metadata), desc="Staging MIDI files")):
    if MAX_FILES and len(manifest_rows) >= MAX_FILES:
        break

    row_dict = row._asdict()
    midi_ref = row_dict[MIDI_COL]
    source_path = locate_midi_file(midi_ref)

    if source_path is None:
        missing_rows.append({
            "midi_ref": midi_ref,
            "reason": "not_found"
        })
        continue

    rel_path = Path(str(midi_ref).strip().replace("\\", "/"))
    if rel_path.is_absolute() or not rel_path.parts:

        if YEAR_COL is not None and YEAR_COL in row_dict and pd.notna(row_dict[YEAR_COL]):
            rel_path = Path(str(int(row_dict[YEAR_COL]))) / source_path.name
        else:
            rel_path = Path(source_path.name)

    target_path = RAW_MIDI_DIR / rel_path
    stage_mode = stage_file(source_path, target_path)

    manifest_rows.append({
        **row_dict,
        "source_path": str(source_path),
        "raw_midi_path": str(target_path),
        "stage_mode": stage_mode,
    })

manifest = pd.DataFrame(manifest_rows)

if manifest.empty:
    raise RuntimeError(
        "No MIDI files were found. Check whether the Kaggle input contains the MIDI files referenced by the CSV."
    )

manifest_path = PROCESSED_DIR / "maestro_manifest_resolved.csv"
manifest.to_csv(manifest_path, index=False)

if missing_rows:
    pd.DataFrame(missing_rows).to_csv(PROCESSED_DIR / "missing_midi_files.csv", index=False)

print("Resolved MIDI files:", len(manifest))
print("Missing MIDI files:", len(missing_rows))
print("Saved manifest:", manifest_path)
print(manifest.head(3).to_string(index=False))


In [ ]:
from src.preprocessing.midi_parser import MIDIParser

parser = MIDIParser(root_dir=RAW_MIDI_DIR)
dataset = parser.build_metadata()

dataset_df = pd.DataFrame(dataset)
dataset_df.to_csv(PROCESSED_DIR / "parsed_metadata.csv", index=False)

print(f"Parsed MIDI files: {len(dataset_df)}")
display_cols = [c for c in ["path", "genre", "tempo", "duration", "notes_count"] if c in dataset_df.columns]
print(dataset_df[display_cols].head().to_string(index=False))


In [ ]:

split_name = SPLIT_COL or "split"

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

if split_name in manifest.columns:
    split_counts = manifest[split_name].fillna("unknown").value_counts().sort_index()
    axes[0].bar(split_counts.index.astype(str), split_counts.values)
    axes[0].set_title("Files per split")
    axes[0].set_xlabel("Split")
    axes[0].set_ylabel("Count")
else:
    axes[0].axis("off")

if YEAR_COL and YEAR_COL in manifest.columns:
    year_counts = manifest[YEAR_COL].fillna("unknown").astype(str).value_counts().sort_index()
    axes[1].bar(year_counts.index.astype(str), year_counts.values)
    axes[1].set_title("Files per year")
    axes[1].set_xlabel("Year")
    axes[1].set_ylabel("Count")
    axes[1].tick_params(axis="x", rotation=45)
else:
    axes[1].axis("off")

if DURATION_COL and DURATION_COL in manifest.columns:
    durations = pd.to_numeric(manifest[DURATION_COL], errors="coerce").dropna()
    axes[2].hist(durations / 60.0, bins=40)
    axes[2].set_title("Duration distribution")
    axes[2].set_xlabel("Minutes")
    axes[2].set_ylabel("Count")
else:
    axes[2].axis("off")

plt.tight_layout()
plt.savefig(PLOTS_DIR / "metadata_overview.png", dpi=150, bbox_inches="tight")
plt.show()

# Plot 2: note / pitch / velocity statistics on a compact sample to keep Kaggle runs practical.
sample_size = min(64, len(manifest))
sample_manifest = manifest.sample(sample_size, random_state=42) if sample_size else manifest.head(0)

import pretty_midi

pitch_counts = Counter()
velocity_vals = []
note_counts = []

for midi_path in tqdm(sample_manifest["raw_midi_path"].tolist(), desc="Collecting note stats"):
    try:
        midi = pretty_midi.PrettyMIDI(midi_path)
        total_notes = 0
        for instrument in midi.instruments:
            if instrument.is_drum:
                continue
            for note in instrument.notes:
                pitch_counts[note.pitch] += 1
                velocity_vals.append(note.velocity)
                total_notes += 1
        note_counts.append(total_notes)
    except Exception as e:
        print(f"[WARNING] Could not parse {midi_path}: {e}")

fig, axes = plt.subplots(2, 2, figsize=(16, 8))

if pitch_counts:
    pitches = sorted(pitch_counts.keys())
    axes[0, 0].bar(pitches, [pitch_counts[p] for p in pitches])
    axes[0, 0].set_title("Pitch distribution (sample)")
    axes[0, 0].set_xlabel("Pitch")
    axes[0, 0].set_ylabel("Count")
else:
    axes[0, 0].axis("off")

if velocity_vals:
    axes[0, 1].hist(velocity_vals, bins=32)
    axes[0, 1].set_title("Velocity distribution (sample)")
    axes[0, 1].set_xlabel("Velocity")
    axes[0, 1].set_ylabel("Count")
else:
    axes[0, 1].axis("off")

if note_counts:
    axes[1, 0].hist(note_counts, bins=32)
    axes[1, 0].set_title("Notes per file (sample)")
    axes[1, 0].set_xlabel("Notes")
    axes[1, 0].set_ylabel("Count")
else:
    axes[1, 0].axis("off")

if sample_size and "raw_midi_path" in sample_manifest.columns:
    axes[1, 1].text(0.02, 0.8, f"Sampled files: {sample_size}", fontsize=12)
    axes[1, 1].text(0.02, 0.6, f"Missing files: {len(missing_rows)}", fontsize=12)
    axes[1, 1].text(0.02, 0.4, f"Resolved files: {len(manifest)}", fontsize=12)
    axes[1, 1].axis("off")

plt.tight_layout()
plt.savefig(PLOTS_DIR / "pitch_velocity_note_stats.png", dpi=150, bbox_inches="tight")
plt.show()

print("Plots saved to:", PLOTS_DIR)


In [ ]:
from src.preprocessing.piano_roll import build_piano_roll_dataset, piano_roll_to_midi

# Use the dataset's own split column when available, otherwise create a deterministic split.
if SPLIT_COL and SPLIT_COL in manifest.columns:
    split_map = {
        "train": ["train"],
        "val": ["validation", "val", "valid"],
        "test": ["test"],
    }
    split_paths = {}
    for split_name, aliases in split_map.items():
        rows = manifest[manifest[SPLIT_COL].astype(str).str.lower().isin(aliases)]
        split_paths[split_name] = rows["raw_midi_path"].tolist()
else:
    rng = np.random.default_rng(42)
    indices = rng.permutation(len(manifest))
    n_train = int(len(indices) * 0.8)
    n_val = int(len(indices) * 0.1)
    split_paths = {
        "train": manifest.iloc[indices[:n_train]]["raw_midi_path"].tolist(),
        "val": manifest.iloc[indices[n_train:n_train + n_val]]["raw_midi_path"].tolist(),
        "test": manifest.iloc[indices[n_train + n_val:]]["raw_midi_path"].tolist(),
    }

# Save the split file lists for downstream training scripts.
for split_name, paths in split_paths.items():
    np.save(TRAIN_TEST_SPLIT_DIR / f"{split_name}_files.npy", np.array(paths, dtype=object))
    pd.DataFrame({"raw_midi_path": paths}).to_csv(TRAIN_TEST_SPLIT_DIR / f"{split_name}_files.csv", index=False)

# Build the piano-roll windows and save them per split.
split_windows = {}
for split_name, paths in split_paths.items():
    if not paths:
        print(f"[INFO] No files found for split: {split_name}")
        split_windows[split_name] = np.empty((0, 0, 0), dtype=np.float32)
        continue

    save_path = TRAIN_TEST_SPLIT_DIR / f"{split_name}.npy"
    split_windows[split_name] = build_piano_roll_dataset(paths, save_path=save_path)
    print(f"{split_name}: {split_windows[split_name].shape} -> {save_path}")

# Also store a compact preprocessing summary in outputs/servey_results.
summary = pd.DataFrame([
    {"metric": "resolved_files", "value": len(manifest)},
    {"metric": "missing_files", "value": len(missing_rows)},
    {"metric": "train_windows", "value": int(split_windows.get("train", np.empty((0,))).shape[0])},
    {"metric": "val_windows", "value": int(split_windows.get("val", np.empty((0,))).shape[0])},
    {"metric": "test_windows", "value": int(split_windows.get("test", np.empty((0,))).shape[0])},
])

summary.to_csv(SERVEY_RESULTS_DIR / "preprocessing_summary.csv", index=False)
summary.to_csv(PROCESSED_DIR / "preprocessing_summary.csv", index=False)
print(summary.to_string(index=False))


In [ ]:
# Export a few MIDI files so outputs/generated_midis is populated and the round-trip is validated.
sample_split = "train" if "train" in split_windows and len(split_windows["train"]) else next(iter(split_windows.keys()))
sample_windows = split_windows.get(sample_split, np.empty((0, 0, 0), dtype=np.float32))

export_count = min(3, len(sample_windows))
for i in range(export_count):
    midi = piano_roll_to_midi(sample_windows[i])
    out_path = GENERATED_MIDI_DIR / f"{sample_split}_roundtrip_{i+1}.mid"
    midi.dump_midi(str(out_path))
    print("Saved:", out_path)

print("Generated MIDI folder:", GENERATED_MIDI_DIR)


In [ ]:
# Final sanity check for downstream scripts.
print("Ready for training.")
print("Raw MIDI files   :", RAW_MIDI_DIR)
print("Processed arrays :", TRAIN_TEST_SPLIT_DIR)
print("Plots            :", PLOTS_DIR)
print("Generated MIDIs  :", GENERATED_MIDI_DIR)
print("Survey results   :", SERVEY_RESULTS_DIR)
